# Preprocesamiento de Datos

Transforma el dataset crudo extraido de los boletines en un dataset listo para entrenar modelos de IA.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re
import difflib

In [2]:
def preprocesar_datos(df):
    estadisticas = {
        "registros_originales": len(df),
        "productos_originales": df["producto_raw"].nunique(),
        "provincias": df["provincia"].nunique(),
    }

    df_limpio = df[df["estado_precio"] == "completo"].copy()
    df_limpio["producto_raw"] = df_limpio["producto_raw"].apply(normalizar_producto)
    mapa_canonico = construir_mapa_canonico_v2(df_limpio, columna_producto="producto_raw", umbral_base=0.90)
    df_limpio["producto_raw"] = df_limpio["producto_raw"].map(mapa_canonico)
    estadisticas["registros_completos"] = len(df_limpio)
    estadisticas["registros_parciales"] = len(df[df["estado_precio"] == "parcial"])
    estadisticas["registros_invalidos"] = len(df[df["estado_precio"] == "invalido"])
    mapa_categoria = df_limpio.drop_duplicates("producto_raw").set_index("producto_raw")["categoria"].to_dict()

    if "quincena_id" in df_limpio.columns:
        df_limpio["periodo"] = df_limpio["quincena_id"]
    elif "año" in df_limpio.columns and "quincena" in df_limpio.columns:
        df_limpio["periodo"] = df_limpio["año"].astype(str) + "-" + df_limpio["quincena"].astype(str).str.zfill(2)
    else:
        df_limpio["periodo"] = "desconocido"
        

    df_pivot = df_limpio.pivot_table(
        index=["producto_raw", "provincia"],
        columns="periodo",
        values=["precio_anterior", "precio_actual"],
        aggfunc="first"
    )

    df_pivot.columns = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    periodos_unicos = sorted(df_limpio["periodo"].unique())
    cols_ordenadas = ["producto_raw", "provincia"]
    for p in periodos_unicos:
        col_ant = f"precio_anterior_{p}"
        col_act = f"precio_actual_{p}"
        if col_ant in df_pivot.columns:
            cols_ordenadas.append(col_ant)
        if col_act in df_pivot.columns:
            cols_ordenadas.append(col_act)

    cols_existentes = [c for c in cols_ordenadas if c in df_pivot.columns]
    df_pivot = df_pivot[cols_existentes]

    productos_descartados = []
    productos_mantener = []

    for idx, row in df_pivot.iterrows():
        cols_precio = [c for c in df_pivot.columns if c.startswith("precio_actual_")]
        valores = row[cols_precio]
        total_periodos = len(valores)
        faltantes = valores.isna().sum()
        porcentaje_faltantes = faltantes / total_periodos if total_periodos > 0 else 1

        if porcentaje_faltantes > 0.30:
            productos_descartados.append({
                "producto": row["producto_raw"],
                "provincia": row["provincia"],
                "porcentaje_faltantes": round(porcentaje_faltantes * 100, 1)
            })
        else:
            productos_mantener.append(idx)

    df_pivot = df_pivot.loc[productos_mantener]

    cols_precio = [c for c in df_pivot.columns if c.startswith("precio_")]
    df_pivot[cols_precio] = df_pivot[cols_precio].interpolate(method="linear", axis=1)
    df_pivot[cols_precio] = df_pivot[cols_precio].ffill(axis=1).bfill(axis=1)

    estadisticas["registros_despues_filtro"] = len(df_pivot)
    estadisticas["productos_descartados"] = len(productos_descartados)
    estadisticas["lista_descartados"] = productos_descartados

    # 1. ESTA LÍNEA ES VITAL (Es la que borraste por accidente):
    df_modelo, le_producto, le_provincia = _crear_features(df_pivot, periodos_unicos, mapa_categoria)

    # 2. NUEVO FILTRO ANTIRRUIDO (OCR):
    condicion_real = (df_modelo["variacion_real"] >= -80) & (df_modelo["variacion_real"] <= 150)
    condicion_t2 = df_modelo["variacion_t2_t1"].isna() | ((df_modelo["variacion_t2_t1"] >= -80) & (df_modelo["variacion_t2_t1"] <= 150))
    df_modelo = df_modelo[condicion_real & condicion_t2].copy()

    # 3. ESTADÍSTICAS FINALES Y RETORNO:
    estadisticas["registros_modelo"] = len(df_modelo)
    estadisticas["columnas_modelo"] = list(df_modelo.columns)

    return {
        "dataset_final": df_modelo,
        "dataset_wide": df_pivot,
        "estadisticas": estadisticas,
        "productos_descartados": productos_descartados,
        "le_producto": le_producto,
        "le_provincia": le_provincia,
    }

In [3]:
def _crear_features(df_pivot, periodos_unicos, mapa_categoria=None):
    
    registros = []

    total_filas = len(df_pivot)
    for contador, (idx, row) in enumerate(df_pivot.iterrows()):
        if contador % 25 == 0:
            print(f"Procesando {contador}/{total_filas}...")
        producto = row["producto_raw"]
        provincia = row["provincia"]

        # Construir serie de precios temporal ordenada por este producto+provincia
        serie_precios = []
        for periodo in periodos_unicos:
            col_actual = f"precio_actual_{periodo}"
            if col_actual in df_pivot.columns:
                serie_precios.append((periodo, row.get(col_actual)))
            else:
                serie_precios.append((periodo, np.nan))

        # Calcular promedios moviles con rolling sobre la serie temporal
        precios_series = pd.Series([p[1] for p in serie_precios], index=[p[0] for p in serie_precios])
        pm2 = precios_series.shift(1).rolling(window=2, min_periods=2).mean()
        pm3 = precios_series.shift(1).rolling(window=3, min_periods=3).mean()

        for i, periodo in enumerate(periodos_unicos):
            col_actual = f"precio_actual_{periodo}"
            if col_actual not in df_pivot.columns:
                continue

            precio_actual = row.get(col_actual)
            if pd.isna(precio_actual):
                continue

            precio_t1 = None
            if i > 0:
                col_t1 = f"precio_actual_{periodos_unicos[i-1]}"
                if col_t1 in df_pivot.columns:
                    precio_t1 = row.get(col_t1)

            precio_t2 = None
            if i > 1:
                col_t2 = f"precio_actual_{periodos_unicos[i-2]}"
                if col_t2 in df_pivot.columns:
                    precio_t2 = row.get(col_t2)

            if precio_t1 is not None and not pd.isna(precio_t1):
                registro = {
                    "producto": producto,
                    "provincia": provincia,
                    "periodo": periodo,
                    "precio_actual": precio_actual,
                    "precio_t1": precio_t1,
                    "precio_t2": precio_t2 if precio_t2 is not None else np.nan,
                }

                if precio_t2 is not None and not pd.isna(precio_t2) and precio_t2 > 0:
                    registro["variacion_t2_t1"] = round(((precio_t1 - precio_t2) / precio_t2) * 100, 2)
                else:
                    registro["variacion_t2_t1"] = np.nan

                val_pm2 = pm2.loc[periodo]
                val_pm3 = pm3.loc[periodo]

                if pd.notna(val_pm2) and val_pm2 > 0 and precio_t1 is not None and precio_t1 > 0:
                    registro["distancia_pm2_pct"] = round(((precio_t1 - val_pm2) / val_pm2) * 100, 2)
                else:
                    registro["distancia_pm2_pct"] = np.nan

                if pd.notna(val_pm3) and val_pm3 > 0 and precio_t1 is not None and precio_t1 > 0:
                    registro["distancia_pm3_pct"] = round(((precio_t1 - val_pm3) / val_pm3) * 100, 2)
                else:
                    registro["distancia_pm3_pct"] = np.nan

                try:
                    partes = periodo.split("-")
                    if len(partes) == 3 and partes[2].startswith("Q"):
                        registro["año"] = int(partes[0])
                        registro["mes"] = int(partes[1])
                        registro["quincena"] = int(partes[2][1:])
                    else:
                        registro["año"] = int(partes[0])
                        registro["quincena"] = int(partes[1])
                        registro["mes"] = 1 if int(partes[1]) == 1 else 2
                except (ValueError, IndexError):
                    registro["mes"] = np.nan
                    registro["año"] = np.nan
                    registro["quincena"] = np.nan

                variacion_precio = ((precio_actual - precio_t1) / precio_t1) * 100 if precio_t1 > 0 else 0
                registro["variacion_real"] = round(variacion_precio, 2)

                if variacion_precio > 3:
                    registro["comportamiento"] = "Alza"
                elif variacion_precio < -3:
                    registro["comportamiento"] = "Caída"
                else:
                    registro["comportamiento"] = "Estable"

                registro["categoria"] = mapa_categoria.get(producto, "desconocido") if mapa_categoria else "desconocido"

                registros.append(registro)

    df_modelo = pd.DataFrame(registros)
    df_modelo, le_producto, le_provincia = _codificar_categoricas(df_modelo)
    return df_modelo, le_producto, le_provincia

In [4]:
def _codificar_categoricas(df):
    from sklearn.preprocessing import LabelEncoder
    le_producto = LabelEncoder()
    le_provincia = LabelEncoder()
    df["producto_encoded"] = le_producto.fit_transform(df["producto"])
    df["provincia_encoded"] = le_provincia.fit_transform(df["provincia"])
    # NUEVO: categoria como binaria (0=no_perecedero, 1=perecedero)
    df["categoria_perecedero"] = (df["categoria"] == "perecedero").astype(int)
    return df, le_producto, le_provincia

In [5]:
def obtener_resumen(df_modelo):
    resumen = {
        "total_registros": len(df_modelo),
        "productos_unicos": df_modelo["producto"].nunique(),
        "provincias": df_modelo["provincia"].nunique(),
        "periodos": df_modelo["periodo"].nunique(),
        "distribucion_comportamiento": df_modelo["comportamiento"].value_counts().to_dict(),
        "valores_faltantes": df_modelo.isnull().sum().to_dict(),
    }
    return resumen

In [6]:
def normalizar_producto(nombre):
    """
    Limpieza agresiva de nombres de producto para unificar variantes de OCR:
    - Símbolos +, ++, +++ (alarmas mal leídas)
    - Números (precios) pegados al final
    - Espacios faltantes entre número y unidad ("105Ib" -> "105 lb")
    - "Ib"/"ib" mal leído como "lb" (error clásico I/l de OCR)
    - Espacios faltantes entre palabras pegadas ("enVaina" -> "en Vaina")
    - Unidades en minúscula consistente (Kg -> kg)
    - Espacios extra alrededor de paréntesis y guiones
    - Minúsculas para comparación consistente
    """
    if not isinstance(nombre, str):
        return nombre
 
    n = nombre.strip()
 
    # 1. Quitar símbolos + al inicio/fin (alarmas OCR)
    n = re.sub(r"^\+{1,3}\s*", "", n)
    n = re.sub(r"\s*\+{1,3}\s*$", "", n)
 
    # 2. Quitar número (precio) pegado al final tras paréntesis de cierre
    n = re.sub(r"(\))\s+\d+[.,]?\d*\s*$", r"\1", n)
 
    # 3. Insertar espacio entre dígito y letra pegados (ej: "105Ib" -> "105 Ib", "50Kg" -> "50 Kg")
    n = re.sub(r"(\d)([A-Za-z])", r"\1 \2", n)
 
    # 4. Corregir "Ib"/"ib" -> "lb" SOLO cuando sigue a un número (error clásico I<->l del OCR)
    n = re.sub(r"(\d)\s*[Ii]b\b", r"\1 lb", n)
 
    # 5. Insertar espacio faltante entre palabra en minúscula y palabra en mayúscula pegadas
    #    (ej: "enVaina" -> "en Vaina", pero cuidado con acrónimos como "UHT")
    n = re.sub(r"(?<=[a-z])(?=[A-Z][a-z])", " ", n)
 
    # 6. Unificar mayúsculas de unidades comunes a minúscula
    n = re.sub(r"\b(Kg|KG)\b", "kg", n)
    n = re.sub(r"\b(Lb|LB)\b", "lb", n)
    n = re.sub(r"\b(Lt|LT)\b", "lt", n)
 
    # 7. Correcciones de typos conocidos observados en los datos
    correcciones = {
        r"\bTiermna\b": "Tierna",
        r"\bRinion\b": "Rinon",
        r"\bAzuicar\b": "Azucar",
        r"\bAzicar\b": "Azucar",
        r"\bInvemadero\b": "Invernadero",
    }
    for patron, reemplazo in correcciones.items():
        n = re.sub(patron, reemplazo, n, flags=re.IGNORECASE)
 
    # 8. Normalizar espacios alrededor de paréntesis y guiones
    n = re.sub(r"\(\s+", "(", n)
    n = re.sub(r"\s+\)", ")", n)
    n = re.sub(r"\s*-\s*", "-", n)
 
    # 9. Espacio faltante entre "aprox." y el número (ej: "aprox.20" -> "aprox. 20")
    n = re.sub(r"aprox\.(\d)", r"aprox. \1", n, flags=re.IGNORECASE)
 
    # 10. Colapsar espacios múltiples
    n = re.sub(r"\s+", " ", n)
 
    return n.strip()


In [7]:
def agrupar_por_similitud(nombres_unicos, umbral=0.90):
    """
    Segunda capa: agrupa nombres que quedaron casi idénticos después de la
    normalización por reglas, usando similitud de texto (difflib).
    Cada grupo se mapea al nombre MÁS FRECUENTE del grupo (canonical).
 
    Devuelve un diccionario {nombre_variante: nombre_canonico}.
    """
    nombres = sorted(nombres_unicos, key=lambda x: str(x))
    n = len(nombres)
    padre = list(range(n))
 
    def encontrar(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x
 
    def unir(x, y):
        rx, ry = encontrar(x), encontrar(y)
        if rx != ry:
            padre[ry] = rx
 
    # Compara cada par (es O(n^2), aceptable para ~600 nombres, no para decenas de miles)
    for i in range(n):
        for j in range(i + 1, n):
            if difflib.SequenceMatcher(None, str(nombres[i]).lower(), str(nombres[j]).lower()).ratio() >= umbral:
                unir(i, j)
 
    grupos = {}
    for i in range(n):
        raiz = encontrar(i)
        grupos.setdefault(raiz, []).append(nombres[i])
 
    return grupos  # {indice_raiz: [lista de nombres del grupo]}

In [8]:
def construir_mapa_canonico(df, columna_producto="producto_raw", columna_frecuencia=None, umbral=0.90):
    """
    Construye el diccionario final {nombre_variante: nombre_canonico} para
    aplicar sobre todo el dataset. El nombre canónico de cada grupo es el
    MÁS FRECUENTE (el que más veces aparece en los datos crudos).
    """
    conteo = df[columna_producto].value_counts()
    nombres_unicos = conteo.index.tolist()
 
    grupos = agrupar_por_similitud(nombres_unicos, umbral=umbral)
 
    mapa = {}
    for _, variantes in grupos.items():
        # Elegir como canónico el que tiene mayor frecuencia en los datos
        canonico = max(variantes, key=lambda v: conteo.get(v, 0))
        for v in variantes:
            mapa[v] = canonico
 
    return mapa

In [9]:
import re


def separar_nombre_y_presentacion(nombre_normalizado):
    """
    Separa el nombre normalizado en (nombre_base, presentacion).
    Ej: "Tomate Rinon de Invernadero (Carton aprox. 55 lb)"
        -> ("Tomate Rinon de Invernadero", "(Carton aprox. 55 lb)")

    Esto permite agrupar por similitud SOLO el nombre_base (para unir errores
    de OCR como "Rifion"/"Rinon"), sin fusionar presentaciones distintas
    (Carton 55lb vs Gaveta 40lb), que deben seguir siendo series separadas.
    """
    match = re.match(r"^(.*?)(\([^)]*\)?.*)$", nombre_normalizado)
    if match:
        base = match.group(1).strip()
        presentacion = match.group(2).strip()
        return base, presentacion
    return nombre_normalizado.strip(), ""


def construir_mapa_canonico_v2(df, columna_producto="producto_raw", umbral_base=0.90):
    """
    Versión corregida: agrupa por similitud SOLO el nombre base (antes del
    paréntesis de presentación), y luego reconstruye el nombre completo
    canónico + presentación original, para no mezclar presentaciones
    distintas del mismo producto.
    """
    conteo = df[columna_producto].value_counts()
    nombres_unicos = conteo.index.tolist()

    # Separar cada nombre en (base, presentacion)
    partes = {n: separar_nombre_y_presentacion(n) for n in nombres_unicos}
    bases_unicas = sorted(set(p[0] for p in partes.values()))

    # Agrupar SOLO las bases por similitud
    import difflib
    n = len(bases_unicas)
    padre = list(range(n))

    def encontrar(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    def unir(x, y):
        rx, ry = encontrar(x), encontrar(y)
        if rx != ry:
            padre[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if difflib.SequenceMatcher(None, bases_unicas[i].lower(), bases_unicas[j].lower()).ratio() >= umbral_base:
                unir(i, j)

    grupos_base = {}
    for i in range(n):
        raiz = encontrar(i)
        grupos_base.setdefault(raiz, []).append(bases_unicas[i])

    # Frecuencia total de cada base (sumando todas sus presentaciones) para elegir la canónica
    freq_base = {}
    for nombre, (base, _) in partes.items():
        freq_base[base] = freq_base.get(base, 0) + conteo.get(nombre, 0)

    mapa_base_canonica = {}
    for _, variantes_base in grupos_base.items():
        base_canonica = max(variantes_base, key=lambda b: freq_base.get(b, 0))
        for v in variantes_base:
            mapa_base_canonica[v] = base_canonica

    # Reconstruir nombre completo: base_canonica + presentacion original de cada fila
    mapa_final = {}
    for nombre, (base, presentacion) in partes.items():
        base_can = mapa_base_canonica.get(base, base)
        nombre_final = f"{base_can} {presentacion}".strip()
        mapa_final[nombre] = nombre_final

    return mapa_final

In [10]:
df_crudo = pd.read_csv('data/processed/dataset_crudo_sipa.csv')
print(f"Registros crudos cargados: {len(df_crudo)}")

resultado_preproc = preprocesar_datos(df_crudo)
df = resultado_preproc['dataset_final']

print(f"Registros preprocesados (con umbral ±3%): {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nDistribución de comportamiento:")
print(df['comportamiento'].value_counts())

df.to_csv('data/processed/dataset_preprocesado_sipa.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Guardado en: data/processed/dataset_preprocesado_sipa.csv")

Registros crudos cargados: 30088
Procesando 0/118...
Procesando 25/118...
Procesando 50/118...
Procesando 75/118...
Procesando 100/118...
Registros preprocesados (con umbral ±3%): 21580
Columnas: ['producto', 'provincia', 'periodo', 'precio_actual', 'precio_t1', 'precio_t2', 'variacion_t2_t1', 'distancia_pm2_pct', 'distancia_pm3_pct', 'año', 'mes', 'quincena', 'variacion_real', 'comportamiento', 'categoria', 'producto_encoded', 'provincia_encoded', 'categoria_perecedero']

Distribución de comportamiento:
comportamiento
Estable    12033
Alza        4833
Caída       4714
Name: count, dtype: int64

✅ Guardado en: data/processed/dataset_preprocesado_sipa.csv


In [21]:
df_test2 = df_crudo[df_crudo['estado_precio'] == 'completo'].copy()
df_test2['producto_norm2'] = df_test2['producto_raw'].apply(normalizar_producto)

print("Productos únicos crudos:", df_crudo['producto_raw'].nunique())
print("Después de normalizar (reglas):", df_test2['producto_norm2'].nunique())

mapa_canonico = construir_mapa_canonico(df_test2, columna_producto='producto_norm2', umbral=0.90)
df_test2['producto_final'] = df_test2['producto_norm2'].map(mapa_canonico)

print("Después de agrupar por similitud:", df_test2['producto_final'].nunique())

Productos únicos crudos: 1254
Después de normalizar (reglas): 322
Después de agrupar por similitud: 86


In [12]:
mapa_v2 = construir_mapa_canonico_v2(df_test2, columna_producto='producto_norm2', umbral_base=0.90)
df_test2['producto_v3'] = df_test2['producto_norm2'].map(mapa_v2)

print("Después de agrupar (respetando presentación):", df_test2['producto_v3'].nunique())

subset_tomate = df_test2[df_test2['producto_v3'].str.contains('Tomate Rinon', case=False, na=False)]
print(subset_tomate['producto_v3'].value_counts())

Después de agrupar (respetando presentación): 225
producto_v3
Tomate Rinon de Invernadero (Gaveta aprox. 40 lb)     184
Tomate Rinon de Invernadero (Carton aprox. 55 lb)     181
Tomate Rinon de Invernadero (Caja aprox. 40 lb)       125
Tomate Rinon de Invernadero (Carton aprox. 40 lb)      57
Tomate Rinon de Invernadero (Carton aprox. 55 lb）)      2
Tomate Rinon de Invernadero（Carton aprox. 55 lb)        1
Tomate Rinon de Invernadero (Caja aprox. 35 lb)         1
Tomate Rinon de Invernadero (Caja aprox. 40 lb）)        1
Name: count, dtype: int64


In [13]:
grupos_raw = agrupar_por_similitud(df_test2['producto_norm2'].unique(), umbral=0.90)

# Mostrar solo los grupos con más de 1 variante (para inspeccionar)
grupos_multiples = {k: v for k, v in grupos_raw.items() if len(v) > 1}
print(f"Total de grupos con múltiples variantes: {len(grupos_multiples)}\n")

for i, (_, variantes) in enumerate(sorted(grupos_multiples.items(), key=lambda x: -len(x[1]))):
    print(f"Grupo {i+1} ({len(variantes)} variantes):")
    for v in variantes:
        print(f"   - {v}")
    print()
    if i >= 20:  # solo muestra los primeros 20 grupos para no saturar
        print("... (más grupos)")
        break

Total de grupos con múltiples variantes: 64

Grupo 1 (21 variantes):
   - Tomate Rifion de Invernadero (Carton aprox. 55 lb)
   - Tomate Rifion de Invernadero (Gaveta aprox. 40 lb)
   - Tomate Rin6 n de Invernadero (Caja aprox. 40 lb)
   - Tomate Rinon de Invermadero (Caja aprox. 40 lb)
   - Tomate Rinon de Invermadero (Carton aprox. 55 lb)
   - Tomate Rinon de Invermadero (Gaveta aprox. 40 lb)
   - Tomate Rinon de Invernadero (Caja aprox. 35 lb)
   - Tomate Rinon de Invernadero (Caja aprox. 40 lb)
   - Tomate Rinon de Invernadero (Carton aprox. 40 lb)
   - Tomate Rinon de Invernadero (Carton aprox. 55 lb)
   - Tomate Rinon de Invernadero (Carton aprox. 55 lb）)
   - Tomate Rinon de Invernadero (Gaveta aprox. 40 lb)
   - Tomate Rinon de Invernadero(Caja aprox. 40 lb)
   - Tomate Rinon de Invernadero(Caja aprox. 40 lb）)
   - Tomate Rinon de Invernadero(Carton aprox. 55 lb)
   - Tomate Rinon de Invernadero(Gaveta aprox. 40 lb)
   - Tomate Rinon de Invernadero（Carton aprox. 55 lb)
   - Tom

In [22]:
# Cuántas combinaciones producto+provincia existen en total con los nombres ya limpios (antes de filtrar por 30%)
total_antes_filtro = resultado_preproc['estadisticas']['registros_despues_filtro'] + resultado_preproc['estadisticas']['productos_descartados']
print("Total de combinaciones ANTES del filtro de 30%:", total_antes_filtro)
print("Sobreviven después del filtro:", resultado_preproc['estadisticas']['registros_despues_filtro'])
print("Descartadas:", resultado_preproc['estadisticas']['productos_descartados'])

Total de combinaciones ANTES del filtro de 30%: 338
Sobreviven después del filtro: 118
Descartadas: 220


In [23]:
dataset_wide = resultado_preproc['dataset_wide']
tomate_rows = dataset_wide[dataset_wide['producto_raw'].str.contains('Tomate Rinon', case=False, na=False)]
print(tomate_rows[['producto_raw', 'provincia']])

# Cobertura real por fila (cuántas columnas de precio_actual tienen dato)
cols_precio = [c for c in dataset_wide.columns if c.startswith('precio_actual_')]
for idx, row in tomate_rows.iterrows():
    cobertura = row[cols_precio].notna().sum()
    print(f"{row['producto_raw']} | {row['provincia']} -> cobertura: {cobertura}/{len(cols_precio)}")

                                          producto_raw provincia
294  Tomate Rinon de Invernadero (Carton aprox. 55 lb)    GUAYAS
296  Tomate Rinon de Invernadero (Gaveta aprox. 40 lb)     AZUAY
Tomate Rinon de Invernadero (Carton aprox. 55 lb) | GUAYAS -> cobertura: 184/184
Tomate Rinon de Invernadero (Gaveta aprox. 40 lb) | AZUAY -> cobertura: 184/184


In [24]:
descartados = pd.DataFrame(resultado_preproc['productos_descartados'])
print(descartados.sort_values('porcentaje_faltantes').head(20))
print(f"\nTotal descartados: {len(descartados)}")
print(f"\nDistribución de % faltante entre los descartados:")
print(descartados['porcentaje_faltantes'].describe())

                                            producto  provincia  \
182  Tomate Rinon de Invernadero (Caja aprox. 40 lb)  PICHINCHA   
84            Maiz Suave Choclo (Saco aprox. 100 lb)      AZUAY   
165                  Pollo Faenado Con Visceras (lb)      AZUAY   
115            Mora de Castilla (Balde aprox. 15 lb)     GUAYAS   
70                   Leche UHT-Entera (Funda de 1 l)  PICHINCHA   
129             Papaya Nacional (Unidad aprox. 7 lb)  PICHINCHA   
69                   Leche UHT-Entera (Funda de 1 l)      AZUAY   
198                           Yogurt (Envase de 1 l)      AZUAY   
123                    Naranja (Ciento aprox. 50 lb)  PICHINCHA   
137             Pepinillo Pepino (Saco aprox. 67 lb)  PICHINCHA   
14               Ajo Bulbo Seco (Malla aprox. 22 lb)      AZUAY   
9              Aguacate Fuerte (Carton aprox. 22 lb)     GUAYAS   
190            Tomate de Arbol (Carton aprox. 22 lb)     GUAYAS   
189            Tomate de Arbol (Carton aprox. 22 lb)      AZUA

In [14]:
%whos DataFrame

Variable        Type         Data/Info
--------------------------------------
df              DataFrame    Shape: (21580, 18)
df_crudo        DataFrame    Shape: (30088, 12)
df_test2        DataFrame    Shape: (27228, 15)
subset_tomate   DataFrame    Shape: (552, 15)


In [15]:
import pandas as pd

# Cargar el dataset crudo (ajusta la ruta si es necesario)
df_crudo = pd.read_csv('data/processed/dataset_crudo_sipa.csv')

# Ejecutar el preprocesamiento real
resultado = preprocesar_datos(df_crudo)
df_modelo = resultado['dataset_final']

print(f"Registros en el dataset final: {len(df_modelo)}")
df_modelo.head()

Procesando 0/118...
Procesando 25/118...
Procesando 50/118...
Procesando 75/118...
Procesando 100/118...
Registros en el dataset final: 21580


,producto,provincia,periodo,precio_actual,precio_t1,precio_t2,variacion_t2_t1,distancia_pm2_pct,distancia_pm3_pct,año,mes,quincena,variacion_real,comportamiento,categoria,producto_encoded,provincia_encoded,categoria_perecedero
0,Aceite Vegetal-Favorita (Caja aprox. 15 I),AZUAY,2018-11-Q2,26.50,26.50,NaN,NaN,NaN,NaN,2018,11,2,0.00,Estable,no_perecedero,0,0,0
1,Aceite Vegetal-Favorita (Caja aprox. 15 I),AZUAY,2018-12-Q1,26.50,26.50,26.5,0.00,0.00,NaN,2018,12,1,0.00,Estable,no_perecedero,0,0,0
2,Aceite Vegetal-Favorita (Caja aprox. 15 I),AZUAY,2018-12-Q2,26.50,26.50,26.5,0.00,0.00,0.00,2018,12,2,0.00,Estable,no_perecedero,0,0,0
3,Aceite Vegetal-Favorita (Caja aprox. 15 I),AZUAY,2019-01-Q1,26.25,26.50,26.5,0.00,0.00,0.00,2019,1,1,-0.94,Estable,no_perecedero,0,0,0
4,Aceite Vegetal-Favorita (Caja aprox. 15 I),AZUAY,2019-01-Q2,26.25,26.25,26.5,-0.94,-0.47,-0.63,2019,1,2,0.00,Estable,no_perecedero,0,0,0


In [16]:
print(df_modelo['comportamiento'].value_counts())
print()
print(df_modelo['comportamiento'].value_counts(normalize=True).round(3) * 100)

comportamiento
Estable    12033
Alza        4833
Caída       4714
Name: count, dtype: int64

comportamiento
Estable    55.8
Alza       22.4
Caída      21.8
Name: proportion, dtype: float64


In [17]:
print(df_crudo['quincena_id'].nunique())
print(sorted(df_crudo['quincena_id'].unique()))

184
['2018-10-Q1', '2018-11-Q2', '2018-12-Q1', '2018-12-Q2', '2019-01-Q1', '2019-01-Q2', '2019-02-Q1', '2019-02-Q2', '2019-03-Q1', '2019-03-Q2', '2019-04-Q1', '2019-04-Q2', '2019-05-Q1', '2019-05-Q2', '2019-06-Q1', '2019-06-Q2', '2019-07-Q1', '2019-07-Q2', '2019-08-Q1', '2019-08-Q2', '2019-09-Q1', '2019-09-Q2', '2019-10-Q1', '2019-10-Q2', '2019-11-Q1', '2019-11-Q2', '2019-12-Q1', '2019-12-Q2', '2020-01-Q1', '2020-01-Q2', '2020-02-Q1', '2020-02-Q2', '2020-03-Q1', '2020-03-Q2', '2020-04-Q1', '2020-04-Q2', '2020-05-Q1', '2020-05-Q2', '2020-06-Q1', '2020-06-Q2', '2020-07-Q1', '2020-07-Q2', '2020-08-Q1', '2020-08-Q2', '2020-09-Q1', '2020-09-Q2', '2020-10-Q1', '2020-10-Q2', '2020-11-Q1', '2020-11-Q2', '2020-12-Q1', '2020-12-Q2', '2021-01-Q1', '2021-01-Q2', '2021-02-Q1', '2021-02-Q2', '2021-03-Q1', '2021-03-Q2', '2021-04-Q1', '2021-04-Q2', '2021-05-Q1', '2021-05-Q2', '2021-06-Q1', '2021-06-Q2', '2021-07-Q1', '2021-07-Q2', '2021-08-Q1', '2021-08-Q2', '2021-09-Q1', '2021-09-Q2', '2021-10-Q1', '

In [18]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)

In [19]:
print("Quincenas únicas:", df_crudo['quincena_id'].nunique())
print(sorted(df_crudo['quincena_id'].unique()))
print("\nProductos únicos en crudo:", df_crudo['producto_raw'].nunique())
print("Combinaciones producto+provincia en el pivot:", resultado_preproc['dataset_wide'].shape[0])
print("Productos descartados por >30% faltante:", resultado_preproc['estadisticas']['productos_descartados'])

Quincenas únicas: 184
['2018-10-Q1', '2018-11-Q2', '2018-12-Q1', '2018-12-Q2', '2019-01-Q1', '2019-01-Q2', '2019-02-Q1', '2019-02-Q2', '2019-03-Q1', '2019-03-Q2', '2019-04-Q1', '2019-04-Q2', '2019-05-Q1', '2019-05-Q2', '2019-06-Q1', '2019-06-Q2', '2019-07-Q1', '2019-07-Q2', '2019-08-Q1', '2019-08-Q2', '2019-09-Q1', '2019-09-Q2', '2019-10-Q1', '2019-10-Q2', '2019-11-Q1', '2019-11-Q2', '2019-12-Q1', '2019-12-Q2', '2020-01-Q1', '2020-01-Q2', '2020-02-Q1', '2020-02-Q2', '2020-03-Q1', '2020-03-Q2', '2020-04-Q1', '2020-04-Q2', '2020-05-Q1', '2020-05-Q2', '2020-06-Q1', '2020-06-Q2', '2020-07-Q1', '2020-07-Q2', '2020-08-Q1', '2020-08-Q2', '2020-09-Q1', '2020-09-Q2', '2020-10-Q1', '2020-10-Q2', '2020-11-Q1', '2020-11-Q2', '2020-12-Q1', '2020-12-Q2', '2021-01-Q1', '2021-01-Q2', '2021-02-Q1', '2021-02-Q2', '2021-03-Q1', '2021-03-Q2', '2021-04-Q1', '2021-04-Q2', '2021-05-Q1', '2021-05-Q2', '2021-06-Q1', '2021-06-Q2', '2021-07-Q1', '2021-07-Q2', '2021-08-Q1', '2021-08-Q2', '2021-09-Q1', '2021-09-Q2